# Task 2: Sentiment Classification on Yelp Polarity

This notebook trains three models from scratch (baseline, BiLSTM, BiLSTM with attention). The config name is read from the `CONFIG` environment variable (`local`, `gpu` or `gpu_200k`).

## 1. Setup

In [ ]:
import os
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.append("src")
from data import ROOT, clean_text, get_data, load_config, load_test_raw

config_name = os.environ.get("CONFIG", "local")
cfg = load_config(config_name)

random.seed(cfg["seed"])
np.random.seed(cfg["seed"])
torch.manual_seed(cfg["seed"])

out_dir = ROOT / cfg["paths"]["output_dir"]
out_dir.mkdir(parents=True, exist_ok=True)
print("config:", config_name)

## 2. Load Data

In [ ]:
train_loader, val_loader, test_loader, word_to_idx, info = get_data(cfg)
print("train reviews:", info["counts"]["train"])
print("val reviews:  ", info["counts"]["val"])
print("test reviews: ", info["counts"]["test"])

## 3. Exploring the Data

### Class balance

In [ ]:
balance = pd.DataFrame(info["class_counts"]).T
balance["positive share"] = (balance["positive"] / (balance["positive"] + balance["negative"])).round(3)
print(balance)

balance[["negative", "positive"]].plot(kind="bar", figsize=(6, 4), rot=0)
plt.title("Class Counts per Split")
plt.xlabel("split")
plt.ylabel("reviews")
plt.savefig(out_dir / "class_balance.png", dpi=150, bbox_inches="tight")
plt.show()

### Missing or broken rows

In [ ]:
dropped = pd.DataFrame(info["dropped"]).T[["total", "not_text", "empty", "bad_label", "kept"]]
print(dropped)
print("empty after cleaning:", info["empty_after_cleaning"])

### Review length

In [ ]:
arrays = np.load(ROOT / cfg["paths"]["processed_dir"] / "arrays.npz")
words = arrays["train_words"]
p50, p90, p95 = np.percentile(words, [50, 90, 95])
print(f"words per review: 50th = {p50:.0f}, 90th = {p90:.0f}, 95th = {p95:.0f}")

plt.figure(figsize=(7, 4))
plt.hist(words, bins=50, range=(0, np.percentile(words, 99)), color="steelblue")
for value, name in [(p50, "50th"), (p90, "90th"), (p95, "95th")]:
    plt.axvline(value, color="red", linestyle="--")
    plt.text(value, plt.ylim()[1] * 0.95, f" {name}: {value:.0f}", rotation=90, va="top")
plt.title("Review Length in Words (train)")
plt.xlabel("words")
plt.ylabel("reviews")
plt.savefig(out_dir / "length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Cleaning and Tokenizing

### Before and after cleaning

In [ ]:
raw = load_test_raw(cfg)
for i in range(3):
    text = raw["text"].iloc[i]
    print(f"review {i + 1} (label {raw['label'].iloc[i]})")
    print("before:", text[:400])
    print("after: ", clean_text(text)[:400])
    print()

### Vocab and max length

In [ ]:
print("vocab size:", info["vocab_size"])
print("max length:", info["max_len"])
for name, share in info["cut_share"].items():
    print(f"{name}: {share * 100:.1f}% of reviews are cut at max length")